# PEFT QLoRA VRAM Measurement

Apply Parameter-Efficient Fine-Tuning (PEFT) using QLoRA (4-bit/8-bit Quantization) on a Small Language Model (SLM) and measure serving VRAM reduction.

In [1]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

e:\Programs\Introduction-to-LLM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No prebuilt binary for CUDA 13.4, loading CUDA 13.2 instead. Set BNB_CUDA_VERSION to override.
W1030 17:12:48.588000 8904 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


## Setup

In [2]:
model_id = "facebook/opt-350m"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def get_vram_mb():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024 ** 2)
    return 0

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

## Baseline SLM (FP16) Loading

In [3]:
clear_vram()
initial_vram = get_vram_mb()

model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

vram_fp16 = get_vram_mb() - initial_vram

del model_fp16
clear_vram()

Loading weights: 100%|██████████| 388/388 [00:00<00:00, 2267.19it/s]


## QLoRA SLM Loading (4-bit)

In [4]:
initial_vram = get_vram_mb()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_qlora = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model_qlora = prepare_model_for_kbit_training(model_qlora)

Loading weights: 100%|██████████| 388/388 [00:00<00:00, 462.93it/s]


## PEFT LoRA Configuration

In [5]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model_qlora, lora_config)

vram_qlora = get_vram_mb() - initial_vram

## VRAM Reduction Comparison

In [6]:
peft_model.print_trainable_parameters()

print("\n" + "="*50)
print("VRAM Utilization Comparison (Serving & Training Prep)")
print("="*50)
print(f"Standard FP16 Model VRAM:  {vram_fp16:.2f} MB")
print(f"QLoRA (4-bit + PEFT) VRAM: {vram_qlora:.2f} MB")
print("-" * 50)

if vram_fp16 > 0:
    reduction = ((vram_fp16 - vram_qlora) / vram_fp16) * 100
    print(f"Total VRAM Reduction:      {reduction:.2f}%")
else:
    print("CUDA not available. VRAM measurements require a GPU.")

trainable params: 786,432 || all params: 331,982,848 || trainable%: 0.2369

VRAM Utilization Comparison (Serving & Training Prep)
Standard FP16 Model VRAM:  632.61 MB
QLoRA (4-bit + PEFT) VRAM: 259.81 MB
--------------------------------------------------
Total VRAM Reduction:      58.93%
